In [ ]:
import numpy as np
from scipy.stats import gaussian_kde, lognorm

def mc_cumprod_samples(field_matrix, simulations="all", years="all",
                       n_samples=10_000, method="bootstrap", random_state=42):
    """
    Generate Monte Carlo samples of cumulative oil production (MMbbl).

    Parameters
    ----------
    field_matrix : pandas.DataFrame
        Index = years; columns = 'simulacao_i' with field production (bbl/year).
    simulations : "all" | "a-b" | list[int]
        Which simulations to use as the empirical base.
    years : "all" | "a-b" | list[int]
        Which years to include before summing to cumulative.
    n_samples : int
        Number of Monte Carlo samples to draw.
    method : {"bootstrap", "lognormal", "kde"}
        - "bootstrap": nonparametric resampling with replacement from empirical cumulative.
        - "lognormal": fit lognormal (MLE) to empirical cumulative and sample.
        - "kde": sample from a Gaussian KDE using inverse CDF on a fine grid.
    random_state : int
        RNG seed for reproducibility.

    Returns
    -------
    samples : np.ndarray, shape (n_samples,)
        Monte Carlo samples of cumulative oil production (MMbbl).
    base_cumulative : np.ndarray
        Empirical cumulative values (MMbbl) used to fit/sample.
    """
    rng = np.random.default_rng(random_state)

    # --- Select simulations ---
    years_idx = field_matrix.index
    if simulations == "all":
        selected_sims = field_matrix.columns.tolist()
    elif isinstance(simulations, str) and "-" in simulations:
        a, b = map(int, simulations.split("-"))
        selected_sims = [f"simulacao_{i}" for i in range(a, b + 1)]
    elif isinstance(simulations, list):
        selected_sims = [f"simulacao_{i}" for i in simulations]
    else:
        raise ValueError("Invalid simulations input.")

    # --- Select years ---
    if years == "all":
        sel = field_matrix.loc[years_idx, selected_sims]
    elif isinstance(years, str) and "-" in years:
        a, b = map(int, years.split("-"))
        sel = field_matrix.loc[a:b, selected_sims]
    elif isinstance(years, list):
        sel = field_matrix.loc[years, selected_sims]
    else:
        raise ValueError("Invalid years input.")

    # --- Empirical cumulative in MMbbl ---
    base_cumulative = (sel.sum(axis=0) / 1e6).values  # positive data

    if method == "bootstrap":
        # Nonparametric bootstrap: sample with replacement from empirical distribution
        idx = rng.integers(0, len(base_cumulative), size=n_samples)
        samples = base_cumulative[idx]

    elif method == "lognormal":
        # Fit lognormal: X ~ LogNormal(s, scale=exp(m)) using MLE on positive data
        # scipy's lognorm parametrization uses shape=s (sigma), scale=exp(mu)
        # Add tiny jitter to avoid zeros
        x = np.clip(base_cumulative, 1e-12, None)
        shape, loc, scale = lognorm.fit(x, floc=0)  # loc fixed at 0
        samples = lognorm.rvs(shape, loc=0, scale=scale, size=n_samples, random_state=rng)

    elif method == "kde":
        # KDE sampling via inverse CDF on a fine grid
        kde = gaussian_kde(base_cumulative)
        x_min, x_max = float(base_cumulative.min()), float(base_cumulative.max())
        # Slightly widen the support
        pad = 0.05 * (x_max - x_min if x_max > x_min else max(x_min, 1.0))
        grid = np.linspace(max(1e-12, x_min - pad), x_max + pad, 4000)
        pdf = kde(grid)
        pdf = np.clip(pdf, 0, None)
        cdf = np.cumsum(pdf)
        cdf /= cdf[-1]
        # Invert CDF by interpolation
        u = rng.random(n_samples)
        samples = np.interp(u, cdf, grid)

    else:
        raise ValueError("method must be 'bootstrap', 'lognormal', or 'kde'.")

    return samples, base_cumulative


In [ ]:
# 1) 10.000 amostras por bootstrap (recomendado para manter a forma empírica)
samples_boot, empirical = mc_cumprod_samples(field_matrix, n_samples=10_000, method="bootstrap")

# 2) Quantis e média para reportar
q05, q50, q95 = np.quantile(samples_boot, [0.05, 0.50, 0.95])
mean_mc = samples_boot.mean()
print(f"Mean={mean_mc:.3f} MMbbl | P5={q05:.3f} | P50={q50:.3f} | P95={q95:.3f}")
